In [10]:
import pandas as pd
from sklearn.feature_selection import mutual_info_classif
from sklearn.preprocessing import MinMaxScaler
from skrebate import ReliefF
import numpy as np

df = pd.read_csv('data_training.csv', parse_dates=['datetime'])

train_start, train_end = '2024-01-01', '2024-03-30'
test_start, test_end = '2024-04-01', '2024-06-30'

treino = df[(df['datetime'] >= train_start) & (df['datetime'] <= train_end)].copy()
validacao = df[(df['datetime'] >= test_start) & (df['datetime'] <= test_end)].copy()

for sub_df in [treino, validacao]:
    sub_df['year'] = sub_df['datetime'].dt.year
    sub_df['month'] = sub_df['datetime'].dt.month
    sub_df['day'] = sub_df['datetime'].dt.day    
    sub_df['hour'] = sub_df['datetime'].dt.hour
    sub_df['minute'] = sub_df['datetime'].dt.minute
    sub_df.drop(columns=['datetime','date','close','open','low','high','volume','average','amount_stock','id_ticker','business'], inplace=True)

def remove_non_numeric(df):
    return df.select_dtypes(include=[np.number])

X_train = treino.drop(columns=['trend'])
y_train = treino['trend']
X_train = remove_non_numeric(X_train)

X_valid = validacao.drop(columns=['trend'])
y_valid = validacao['trend']
X_valid = remove_non_numeric(X_valid)

scaler = MinMaxScaler()
X_trains = scaler.fit_transform(X_train)


In [ ]:
info_gain = mutual_info_classif(X_trains, y_train, random_state= 42)
info_gain_series = pd.Series(info_gain, index=X_train.columns)
info_gain_sorted = info_gain_series.sort_values(ascending=False)

print("Top Information Gain:")
print(info_gain_sorted.head(10))

relief = ReliefF(n_neighbors=100, n_features_to_select=X_train.shape[1])
relief.fit(X_trains, y_train)
relief_scores = relief.feature_importances_
relief_series = pd.Series(relief_scores, index=X_train.columns)
relief_sorted = relief_series.sort_values(ascending=False)

print("Top ReliefF:")
print(relief_sorted.head(10))

Top Information Gain:
day           0.036399
hour          0.018376
NSMA_5        0.012155
NSMA_7        0.005625
Bands_Norm    0.000516
NSMA_3        0.000512
NSMA_9        0.000081
NSMA_11       0.000000
year          0.000000
month         0.000000
dtype: float64
Top ReliefF:
day           0.079995
minute        0.017563
hour          0.016506
Bands_Norm    0.012732
NSMA_11       0.010029
NSMA_9        0.008912
NSMA_7        0.006854
NSMA_5        0.005725
month         0.003752
NSMA_3        0.003274
dtype: float64


In [7]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

def testar_modelo(features):
    rf = RandomForestClassifier(n_estimators=100, random_state=42)
    rf.fit(X_train[features], y_train)
    
    y_pred = rf.predict(X_valid[features])
    acc = accuracy_score(y_valid, y_pred)
    
    print("Acurácia", acc)
    print(classification_report(y_valid, y_pred))
    
    return rf, acc

top5_info = ['day', 'hour', 'NSMA_5', 'NSMA_7', 'Bands_Norm']
print("top 5 do InfoGain")
modelo_info, acc_info = testar_modelo(top5_info)

top5_relief = ['day', 'minute', 'hour', 'Bands_Norm', 'NSMA_11']
print("top 5 do ReliefF")
modelo_relief, acc_relief = testar_modelo(top5_relief)


top 5 do InfoGain
Acurácia 0.5745022479126526
              precision    recall  f1-score   support

         0.0       0.57      0.55      0.56      3055
         1.0       0.58      0.60      0.59      3173

    accuracy                           0.57      6228
   macro avg       0.57      0.57      0.57      6228
weighted avg       0.57      0.57      0.57      6228

top 5 do ReliefF
Acurácia 0.5614964675658317
              precision    recall  f1-score   support

         0.0       0.56      0.53      0.54      3055
         1.0       0.57      0.59      0.58      3173

    accuracy                           0.56      6228
   macro avg       0.56      0.56      0.56      6228
weighted avg       0.56      0.56      0.56      6228

